In [14]:
# Import core PyTorch libraries
import torch
import torch.nn as nn

In [15]:
# Multi-Head Self Attention Module
class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super(SelfAttention, self).__init__()

        self.embed_size = embed_size        # Total embedding dimension
        self.heads = heads                  # Number of attention heads
        self.head_dim = embed_size // heads # Dimension per head

        # Ensure embedding splits evenly across heads
        assert self.head_dim * heads == embed_size

        # Linear layers for Value, Key, Query projections
        self.values = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.keys   = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.queries= nn.Linear(self.head_dim, self.head_dim, bias=False)

        # Final linear layer after concatenating all heads
        self.fc_out = nn.Linear(heads * self.head_dim, embed_size)

    def forward(self, values, keys, query, mask):
        N = query.shape[0]  # Batch size

        # Reshape input into (N, seq_len, heads, head_dim)
        values = values.reshape(N, values.shape[1], self.heads, self.head_dim)
        keys   = keys.reshape(N, keys.shape[1], self.heads, self.head_dim)
        query  = query.reshape(N, query.shape[1], self.heads, self.head_dim)

        # Compute attention scores using dot product
        energy = torch.einsum("nqhd,nkhd->nhqk", [query, keys])

        # Apply mask (for padding or causal masking)
        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e20"))

        # Normalize scores using softmax
        attention = torch.softmax(energy / (self.embed_size ** 0.5), dim=3)

        # Weighted sum of values
        out = torch.einsum("nhql,nlhd->nqhd", [attention, values])

        # Concatenate heads
        out = out.reshape(N, query.shape[1], self.heads * self.head_dim)

        return self.fc_out(out)

In [16]:
# Single Transformer Block (Attention + Feed Forward)
class TransformerBlock(nn.Module):
    def __init__(self, embed_size, heads, dropout, forward_expansion):
        super().__init__()

        self.attention = SelfAttention(embed_size, heads)

        # Layer normalization (stabilizes training)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)

        # Feed Forward Network (FFN)
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_size, forward_expansion * embed_size),
            nn.ReLU(),
            nn.Linear(forward_expansion * embed_size, embed_size),
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, value, key, query, mask):
        # Step 1: Self Attention
        attn = self.attention(value, key, query, mask)

        # Step 2: Add & Norm
        x = self.dropout(self.norm1(attn + query))

        # Step 3: Feed Forward
        forward = self.feed_forward(x)

        # Step 4: Add & Norm again
        return self.dropout(self.norm2(forward + x))

In [17]:
# Encoder: Stacks multiple Transformer blocks
class Encoder(nn.Module):
    def __init__(self, src_vocab_size, embed_size, num_layers, heads,
                 device, forward_expansion, dropout, max_length):

        super().__init__()
        self.device = device

        # Token + Positional embeddings
        self.word_embedding = nn.Embedding(src_vocab_size, embed_size)
        self.position_embedding = nn.Embedding(max_length, embed_size)

        # Stack of Transformer blocks
        self.layers = nn.ModuleList([
            TransformerBlock(embed_size, heads, dropout, forward_expansion)
            for _ in range(num_layers)
        ])

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        N, seq_len = x.shape

        # Create position indices
        positions = torch.arange(0, seq_len).expand(N, seq_len).to(self.device)

        # Combine token + positional embeddings
        out = self.dropout(
            self.word_embedding(x) + self.position_embedding(positions)
        )

        # Pass through all encoder layers
        for layer in self.layers:
            out = layer(out, out, out, mask)

        return out

In [18]:
# Decoder Block (Masked Attention + Encoder-Decoder Attention)
class DecoderBlock(nn.Module):
    def __init__(self, embed_size, heads, dropout, forward_expansion):
        super().__init__()

        # Masked self-attention (prevents future token access)
        self.attention = SelfAttention(embed_size, heads)

        # Normalization layers
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)
        self.norm3 = nn.LayerNorm(embed_size)

        # Cross attention blocks
        self.block1 = TransformerBlock(embed_size, heads, dropout, forward_expansion)
        self.block2 = TransformerBlock(embed_size, heads, dropout, forward_expansion)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask, trg_mask):
        # Step 1: Masked self-attention
        attn = self.attention(x, x, x, trg_mask)
        query = self.dropout(self.norm1(attn + x))

        # Step 2: Encoder-Decoder attention
        out = self.block1(enc_out, enc_out, query, src_mask)
        out = self.dropout(self.norm2(out + query))

        # Step 3: Additional transformation
        out = self.block2(enc_out, enc_out, out, src_mask)

        return self.dropout(self.norm3(out + query))

In [19]:
# Decoder: Generates output sequence
class Decoder(nn.Module):
    def __init__(self, trg_vocab_size, embed_size, num_layers, heads,
                 forward_expansion, dropout, device, max_length):

        super().__init__()
        self.device = device

        # Token + positional embeddings
        self.word_embedding = nn.Embedding(trg_vocab_size, embed_size)
        self.position_embedding = nn.Embedding(max_length, embed_size)

        # Stack of decoder blocks
        self.layers = nn.ModuleList([
            DecoderBlock(embed_size, heads, dropout, forward_expansion)
            for _ in range(num_layers)
        ])

        # Final output layer → vocabulary
        self.fc_out = nn.Linear(embed_size, trg_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask, trg_mask):
        N, seq_len = x.shape

        positions = torch.arange(0, seq_len).expand(N, seq_len).to(self.device)

        x = self.dropout(
            self.word_embedding(x) + self.position_embedding(positions)
        )

        # Pass through decoder layers
        for layer in self.layers:
            x = layer(x, enc_out, src_mask, trg_mask)

        return self.fc_out(x)

In [20]:
# Full Transformer Model (Encoder + Decoder)
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size,
                 src_pad_idx, trg_pad_idx,
                 embed_size=256, num_layers=6,
                 forward_expansion=4, heads=8,
                 dropout=0, device="cuda", max_length=100):

        super().__init__()

        # Encoder and Decoder
        self.encoder = Encoder(
            src_vocab_size, embed_size, num_layers, heads,
            device, forward_expansion, dropout, max_length
        )

        self.decoder = Decoder(
            trg_vocab_size, embed_size, num_layers, heads,
            forward_expansion, dropout, device, max_length
        )

        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx
        self.device = device

    # Mask to ignore padding tokens
    def make_src_mask(self, src):
        return (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2).to(self.device)

    # Mask to prevent future token leakage
    def make_trg_mask(self, trg):
        N, trg_len = trg.shape
        mask = torch.tril(torch.ones(trg_len, trg_len))
        return mask.expand(N, 1, trg_len, trg_len).to(self.device)

    def forward(self, src, trg):
        src_mask = self.make_src_mask(src)
        trg_mask = self.make_trg_mask(trg)

        enc = self.encoder(src, src_mask)
        return self.decoder(trg, enc, src_mask, trg_mask)

In [21]:
# Device setup (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dummy input data (batch_size=2)
x = torch.tensor([
    [1,5,6,4,3,9,5,2,0],
    [1,8,7,3,4,5,6,7,2]
]).to(device)

trg = torch.tensor([
    [1,7,4,3,5,9,2,0],
    [1,5,6,2,4,7,6,2]
]).to(device)

# Initialize model
model = Transformer(
    src_vocab_size=10,
    trg_vocab_size=10,
    src_pad_idx=0,
    trg_pad_idx=0
).to(device)

# Forward pass (teacher forcing)
out = model(x, trg[:, :-1])

# Output shape → (batch_size, seq_len, vocab_size)
print(out.shape)

torch.Size([2, 7, 10])


In [22]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# 1. Create a simple vocabulary
# -----------------------------
vocab = {
    "<pad>": 0,
    "<s>": 1,
    "i": 2,
    "love": 3,
    "deep": 4,
    "learning": 5,
    "j": 6,
    "aime": 7,
    ".": 8,
    "<eos>": 9
}

# Reverse vocab (for decoding output later)
inv_vocab = {v: k for k, v in vocab.items()}

# -----------------------------
# 2. Convert words → indices
# -----------------------------
# Source sentence: "i love deep learning"
src_sentence = ["<s>", "i", "love", "deep", "learning", ".", "<pad>", "<pad>", "<pad>"]

# Target sentence: "j aime deep learning"
trg_sentence = ["<s>", "j", "aime", "deep", "learning", ".", "<eos>", "<pad>"]

# Convert to numbers
x = torch.tensor([[vocab[word] for word in src_sentence]]).to(device)
trg = torch.tensor([[vocab[word] for word in trg_sentence]]).to(device)

# -----------------------------
# 3. Initialize model
# -----------------------------
model = Transformer(
    src_vocab_size=len(vocab),
    trg_vocab_size=len(vocab),
    src_pad_idx=vocab["<pad>"],
    trg_pad_idx=vocab["<pad>"]
).to(device)

# -----------------------------
# 4. Forward pass
# -----------------------------
out = model(x, trg[:, :-1])

print("Output shape:", out.shape)

# -----------------------------
# 5. Convert predictions → words
# -----------------------------
pred_tokens = out.argmax(dim=2)  # pick highest probability

print("\nPredicted token IDs:", pred_tokens)

# Convert to words
pred_words = [inv_vocab[idx.item()] for idx in pred_tokens[0]]
print("Predicted words:", pred_words)

Output shape: torch.Size([1, 7, 10])

Predicted token IDs: tensor([[5, 7, 5, 4, 5, 3, 5]], device='cuda:0')
Predicted words: ['learning', 'aime', 'learning', 'deep', 'learning', 'love', 'learning']


In [23]:
# ================================
# 1. Imports
# ================================
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================================
# 2. Tiny Dataset
# ================================
pairs = [
    ("i love deep learning", "j aime deep learning"),
    ("i love learning", "j aime learning"),
    ("i love ai", "j aime ai"),
    ("deep learning is fun", "deep learning est fun"),
]

# ================================
# 3. Build Vocabulary
# ================================
special = ["<pad>", "<s>", "<eos>"]

words = set()
for s, t in pairs:
    words.update(s.split())
    words.update(t.split())

vocab = special + sorted(list(words))
stoi = {w:i for i,w in enumerate(vocab)}
itos = {i:w for w,i in stoi.items()}

vocab_size = len(vocab)

# ================================
# 4. Encode Function
# ================================
def encode(sent, max_len=10):
    tokens = ["<s>"] + sent.split() + ["<eos>"]
    ids = [stoi[t] for t in tokens]
    ids += [stoi["<pad>"]] * (max_len - len(ids))
    return ids[:max_len]

data = [(encode(s), encode(t)) for s,t in pairs]

# ================================
# 5. Minimal Transformer
# ================================
class Transformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, 64)
        self.pos = nn.Embedding(20, 64)
        self.fc = nn.Linear(64, vocab_size)

    def forward(self, x, y):
        N, L = y.shape
        pos = torch.arange(0, L).expand(N, L).to(device)
        x = self.embed(x)
        y = self.embed(y) + self.pos(pos)
        return self.fc(y)

model = Transformer().to(device)

# ================================
# 6. Training
# ================================
opt = optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss(ignore_index=stoi["<pad>"])

for epoch in range(200):
    total = 0
    for s, t in data:
        src = torch.tensor([s]).to(device)
        trg = torch.tensor([t]).to(device)

        opt.zero_grad()
        out = model(src, trg[:, :-1])

        out = out.reshape(-1, vocab_size)
        target = trg[:, 1:].reshape(-1)

        loss = loss_fn(out, target)
        loss.backward()
        opt.step()
        total += loss.item()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {total:.3f}")

# ================================
# 7. Translation Function
# ================================
def translate(sent):
    model.eval()
    src = torch.tensor([encode(sent)]).to(device)

    trg = [stoi["<s>"]]

    for _ in range(10):
        trg_tensor = torch.tensor([trg]).to(device)
        out = model(src, trg_tensor)

        next_token = out.argmax(2)[:, -1].item()
        trg.append(next_token)

        if next_token == stoi["<eos>"]:
            break

    return " ".join([itos[i] for i in trg])

# ================================
# 8. Test
# ================================
print("\nTranslations:")
print("i love deep learning →", translate("i love deep learning"))
print("i love ai →", translate("i love ai"))
print("deep learning is fun →", translate("deep learning is fun"))

Epoch 0, Loss: 10.879
Epoch 50, Loss: 1.310
Epoch 100, Loss: 1.303
Epoch 150, Loss: 1.300

Translations:
i love deep learning → <s> j aime ai <eos>
i love ai → <s> j aime ai <eos>
deep learning is fun → <s> j aime ai <eos>


In [24]:
# ================================
# 1. Imports
# ================================
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================================
# 2. Dataset
# ================================
pairs = [
    ("i love deep learning", "j aime deep learning"),
    ("i love learning", "j aime learning"),
    ("i love ai", "j aime ai"),
    ("deep learning is fun", "deep learning est fun"),
]

# ================================
# 3. Vocabulary
# ================================
special = ["<pad>", "<s>", "<eos>"]

words = set()
for s, t in pairs:
    words.update(s.split())
    words.update(t.split())

vocab = special + sorted(list(words))
stoi = {w:i for i,w in enumerate(vocab)}
itos = {i:w for w,i in stoi.items()}
vocab_size = len(vocab)

# ================================
# 4. Encode
# ================================
def encode(sent, max_len=10):
    tokens = ["<s>"] + sent.split() + ["<eos>"]
    ids = [stoi[t] for t in tokens]
    ids += [stoi["<pad>"]] * (max_len - len(ids))
    return ids[:max_len]

data = [(encode(s), encode(t)) for s,t in pairs]

# ================================
# 5. MODEL (YOUR FULL TRANSFORMER)
# ================================
class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super().__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads

        self.values = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.keys   = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.queries= nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.fc_out = nn.Linear(heads * self.head_dim, embed_size)

    def forward(self, values, keys, query, mask):
        N = query.shape[0]

        values = values.reshape(N, values.shape[1], self.heads, self.head_dim)
        keys   = keys.reshape(N, keys.shape[1], self.heads, self.head_dim)
        query  = query.reshape(N, query.shape[1], self.heads, self.head_dim)

        energy = torch.einsum("nqhd,nkhd->nhqk", [query, keys])

        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e20"))

        attention = torch.softmax(energy / (self.embed_size ** 0.5), dim=3)

        out = torch.einsum("nhql,nlhd->nqhd", [attention, values])
        out = out.reshape(N, query.shape[1], self.heads * self.head_dim)

        return self.fc_out(out)


class TransformerBlock(nn.Module):
    def __init__(self, embed_size, heads, dropout, expansion):
        super().__init__()
        self.attn = SelfAttention(embed_size, heads)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)

        self.ff = nn.Sequential(
            nn.Linear(embed_size, expansion * embed_size),
            nn.ReLU(),
            nn.Linear(expansion * embed_size, embed_size),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, v, k, q, mask):
        attn = self.attn(v, k, q, mask)
        x = self.dropout(self.norm1(attn + q))
        f = self.ff(x)
        return self.dropout(self.norm2(f + x))


class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, layers, heads, device):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.pos   = nn.Embedding(20, embed_size)
        self.layers = nn.ModuleList([
            TransformerBlock(embed_size, heads, 0, 4)
            for _ in range(layers)
        ])
        self.device = device

    def forward(self, x, mask):
        N, L = x.shape
        pos = torch.arange(0, L).expand(N, L).to(self.device)
        out = self.embed(x) + self.pos(pos)

        for layer in self.layers:
            out = layer(out, out, out, mask)
        return out


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, layers, heads, device):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.pos   = nn.Embedding(20, embed_size)
        self.layers = nn.ModuleList([
            TransformerBlock(embed_size, heads, 0, 4)
            for _ in range(layers)
        ])
        self.fc = nn.Linear(embed_size, vocab_size)
        self.device = device

    def forward(self, x, enc_out, src_mask, trg_mask):
        N, L = x.shape
        pos = torch.arange(0, L).expand(N, L).to(self.device)
        x = self.embed(x) + self.pos(pos)

        for layer in self.layers:
            x = layer(enc_out, enc_out, x, src_mask)

        return self.fc(x)


class Transformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder(vocab_size, 128, 2, 4, device)
        self.decoder = Decoder(vocab_size, 128, 2, 4, device)

    def forward(self, src, trg):
        src_mask = (src != stoi["<pad>"]).unsqueeze(1).unsqueeze(2)
        trg_mask = torch.tril(torch.ones(trg.shape[1], trg.shape[1])).to(device)

        enc = self.encoder(src, src_mask)
        return self.decoder(trg, enc, src_mask, trg_mask)


model = Transformer().to(device)

# ================================
# 6. Training
# ================================
opt = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss(ignore_index=stoi["<pad>"])

for epoch in range(300):
    total = 0
    for s, t in data:
        src = torch.tensor([s]).to(device)
        trg = torch.tensor([t]).to(device)

        opt.zero_grad()
        out = model(src, trg[:, :-1])

        out = out.reshape(-1, vocab_size)
        target = trg[:, 1:].reshape(-1)

        loss = loss_fn(out, target)
        loss.backward()
        opt.step()
        total += loss.item()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {total:.3f}")

# ================================
# 7. Translate
# ================================
def translate(sent):
    model.eval()
    src = torch.tensor([encode(sent)]).to(device)

    trg = [stoi["<s>"]]

    for _ in range(10):
        trg_tensor = torch.tensor([trg]).to(device)
        out = model(src, trg_tensor)

        next_token = out.argmax(2)[:, -1].item()
        trg.append(next_token)

        if next_token == stoi["<eos>"]:
            break

    return " ".join([itos[i] for i in trg])


# ================================
# 8. Test
# ================================
print("\nTranslations:")
print("i love deep learning →", translate("i love deep learning"))
print("i love ai →", translate("i love ai"))
print("deep learning is fun →", translate("deep learning is fun"))

Epoch 0, Loss: 9.485
Epoch 50, Loss: 0.013
Epoch 100, Loss: 0.006
Epoch 150, Loss: 0.003
Epoch 200, Loss: 0.002
Epoch 250, Loss: 0.002

Translations:
i love deep learning → <s> j aime deep learning <eos>
i love ai → <s> j aime ai <eos>
deep learning is fun → <s> deep learning est fun <eos>
